 -- This notebook targets to quantify and demonstrate evidence of consistency on experts responses on each of the thre questions each assessed differently based on the nature of questions:
 - Feasibility: assess on benchmarck stability in rating using median/proportions,
 - Importance: complete rating distributions and percentage agreements on the 11 DC solutions,
 - Barrier select: assess on agreement between the selected sets among he 11 barriers.

   

** Stability indicator: 
1. feasibility responses: leave-one out still produces same median and IQRs
2. importance responses: medians are stable, if most ratings still fall within one point of the solution median, and overall ordinal agreement coefficient is reported with uncertainty.
3. barrier-select: meaningful overlap in selected sets, and highest-ranked barriers remain stable when individual experts are removed. 

In [2]:
# F1: feasibility (median and IQR)
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/feasibility_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
ratings = pd.to_numeric(df["rating"], errors="coerce").dropna()

feasibility_location = pd.DataFrame({
    "n_experts": [len(ratings)],
    "median": [ratings.median()],
    "q1": [ratings.quantile(0.25)],
    "q3": [ratings.quantile(0.75)],
    "iqr": [ratings.quantile(0.75) - ratings.quantile(0.25)]
})

display(feasibility_location)

# save the results to a CSV file
OUTPUT = ROOT / "experts-consistency/output/"
create_output_dir = OUTPUT.mkdir(parents=True, exist_ok=True)
feasibility_location.to_csv(OUTPUT / "feasibility_location.csv", index=False)

,n_experts,median,q1,q3,iqr
0,22,3.0,3.0,3.75,0.75


In [3]:
# F2 — Proportion rating feasibility ≥ 3
from operator import index
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/feasibility_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
ratings = pd.to_numeric(df["rating"], errors="coerce").dropna()

n_experts = len(ratings)
n_at_least_feasible = int((ratings >= 3).sum())
proportion_at_least_feasible = n_at_least_feasible / n_experts

feasibility_proportion = pd.DataFrame({
    "n_experts": [n_experts],
    "n_rating_ge_3": [n_at_least_feasible],
    "proportion_rating_ge_3": [proportion_at_least_feasible],
    "percentage_rating_ge_3": [100 * proportion_at_least_feasible]
})

display(feasibility_proportion)

# save the results to a CSV file
feasibility_proportion.to_csv(OUTPUT / "feasibility_g3_proportions.csv", index=False)

,n_experts,n_rating_ge_3,proportion_rating_ge_3,percentage_rating_ge_3
0,22,22,1.0,100.0


In [5]:
# F3 — Leave-one-expert-out stability # loo = leave-one-out
# The substantive conclusion that this assessment brings is defined here as: median rating ≥ 3; and at least 50% of experts rate feasibility ≥ 3.
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/feasibility_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["row_id", "rating"]).reset_index(drop=True)

full_median = df["rating"].median()
full_proportion = (df["rating"] >= 3).mean()

full_conclusion = (
    (full_median >= 3) and
    (full_proportion >= 0.50)
)

loo_results = []

for removed_index, removed_row in df.iterrows():
    remaining = df.drop(index=removed_index)
    
    loo_median = remaining["rating"].median()
    loo_proportion = (remaining["rating"] >= 3).mean()
    
    loo_conclusion = (
        (loo_median >= 3) and
        (loo_proportion >= 0.50)
    )
    
    loo_results.append({
        "removed_expert": removed_row["row_id"],
        "removed_rating": removed_row["rating"],
        "loo_n_experts": len(remaining),
        "loo_median": loo_median,
        "loo_proportion_ge_3": loo_proportion,
        "median_changed": not np.isclose(loo_median, full_median),
        "substantive_conclusion": loo_conclusion,
        "conclusion_changed": loo_conclusion != full_conclusion
    })

loo_feasibility = pd.DataFrame(loo_results)

stability_summary = pd.DataFrame({
    "full_sample_median": [full_median],
    "full_sample_proportion_ge_3": [full_proportion],
    "full_sample_conclusion": [full_conclusion],
    "minimum_loo_median": [loo_feasibility["loo_median"].min()],
    "maximum_loo_median": [loo_feasibility["loo_median"].max()],
    "minimum_loo_proportion_ge_3": [
        loo_feasibility["loo_proportion_ge_3"].min()
    ],
    "maximum_loo_proportion_ge_3": [
        loo_feasibility["loo_proportion_ge_3"].max()
    ],
    "n_removals_changing_median": [
        loo_feasibility["median_changed"].sum()
    ],
    "n_removals_changing_conclusion": [
        loo_feasibility["conclusion_changed"].sum()
    ],
    "stable_under_all_removals": [
        not loo_feasibility["conclusion_changed"].any()
    ]
})

display(stability_summary)
display(loo_feasibility)

# save the results to CSV files
loo_feasibility.to_csv(OUTPUT / "loo_feasibility_details.csv", index=False)

,full_sample_median,full_sample_proportion_ge_3,full_sample_conclusion,minimum_loo_median,maximum_loo_median,minimum_loo_proportion_ge_3,maximum_loo_proportion_ge_3,n_removals_changing_median,n_removals_changing_conclusion,stable_under_all_removals
0,3.0,1.0,True,3.0,3.0,1.0,1.0,0,0,True


,removed_expert,removed_rating,loo_n_experts,loo_median,loo_proportion_ge_3,median_changed,substantive_conclusion,conclusion_changed
0,he01_s1,3,21,3.0,1.0,False,True,False
1,he02_s1,3,21,3.0,1.0,False,True,False
2,he03_s1,3,21,3.0,1.0,False,True,False
3,he04_s1,4,21,3.0,1.0,False,True,False
4,he05_s1,4,21,3.0,1.0,False,True,False
5,he06_s1,3,21,3.0,1.0,False,True,False
6,he07_s1,3,21,3.0,1.0,False,True,False
7,he08_s1,3,21,3.0,1.0,False,True,False
8,he09_s1,3,21,3.0,1.0,False,True,False
9,he10_s1,3,21,3.0,1.0,False,True,False


In [ ]:
# F4 — Optional bootstrap CI for proportion ≥ 3
# With the current data, the estimated proportion and both confidence limits will be 1.00 because every expert selected either 3 or 4.
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/feasibility_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
ratings = pd.to_numeric(df["rating"], errors="coerce").dropna().to_numpy()

N_BOOTSTRAP = 10_000
RANDOM_SEED = 20260728

rng = np.random.default_rng(RANDOM_SEED)

bootstrap_proportions = np.empty(N_BOOTSTRAP)

for bootstrap_index in range(N_BOOTSTRAP):
    bootstrap_sample = rng.choice(
        ratings,
        size=len(ratings),
        replace=True
    )
    
    bootstrap_proportions[bootstrap_index] = (
        bootstrap_sample >= 3
    ).mean()

ci_lower, ci_upper = np.quantile(
    bootstrap_proportions,
    [0.025, 0.975]
)

bootstrap_result = pd.DataFrame({
    "observed_proportion_ge_3": [(ratings >= 3).mean()],
    "bootstrap_ci_lower_95": [ci_lower],
    "bootstrap_ci_upper_95": [ci_upper],
    "n_bootstrap_samples": [N_BOOTSTRAP]
})

display(bootstrap_result)

In [ ]:
# B. Importance
# I1 — Median and IQR for each solution
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/importance_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
df["rating_numeric"] = pd.to_numeric(
    df["rating_numeric"],
    errors="coerce"
)

df = df.dropna(
    subset=["respondent_id", "technology", "rating_numeric"]
)

importance_location = (
    df.groupby("technology")["rating_numeric"]
      .agg(
          n_experts="count",
          median="median",
          q1=lambda x: x.quantile(0.25),
          q3=lambda x: x.quantile(0.75)
      )
      .reset_index()
)

importance_location["iqr"] = (
    importance_location["q3"] - importance_location["q1"]
)

display(importance_location)

In [ ]:
# I2 — Modal rating and modal-rating percentage. This cell handles the possibility of tied modes.
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/importance_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
df["rating_numeric"] = pd.to_numeric(
    df["rating_numeric"],
    errors="coerce"
)

df = df.dropna(
    subset=["respondent_id", "technology", "rating_numeric"]
)

modal_results = []

for solution, solution_df in df.groupby("technology"):
    ratings = solution_df["rating_numeric"]
    rating_counts = ratings.value_counts().sort_index()
    
    modal_count = int(rating_counts.max())
    modal_ratings = rating_counts[
        rating_counts == modal_count
    ].index.tolist()
    
    modal_results.append({
        "technology": solution,
        "n_experts": len(ratings),
        "modal_rating": ", ".join(
            str(int(rating)) for rating in modal_ratings
        ),
        "n_selecting_modal_rating": modal_count,
        "modal_rating_proportion": modal_count / len(ratings),
        "modal_rating_percentage": 100 * modal_count / len(ratings)
    })

importance_modes = pd.DataFrame(modal_results)

display(importance_modes)

In [ ]:
# I3 — Exact percentage agreement. This calculates the probability that two experts selected at random from the raters of a solution gave exactly the same rating.
from pathlib import Path
from math import comb
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/importance_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
df["rating_numeric"] = pd.to_numeric(
    df["rating_numeric"],
    errors="coerce"
)

df = df.dropna(
    subset=["respondent_id", "technology", "rating_numeric"]
)

agreement_results = []

for solution, solution_df in df.groupby("technology"):
    ratings = solution_df["rating_numeric"]
    rating_counts = ratings.value_counts()
    n_experts = len(ratings)
    
    total_expert_pairs = comb(n_experts, 2)
    
    exactly_agreeing_pairs = sum(
        comb(int(count), 2)
        for count in rating_counts
        if count >= 2
    )
    
    exact_agreement = (
        exactly_agreeing_pairs / total_expert_pairs
        if total_expert_pairs > 0
        else float("nan")
    )
    
    agreement_results.append({
        "technology": solution,
        "n_experts": n_experts,
        "total_expert_pairs": total_expert_pairs,
        "exactly_agreeing_pairs": exactly_agreeing_pairs,
        "exact_agreement_proportion": exact_agreement,
        "exact_agreement_percentage": 100 * exact_agreement
    })

importance_exact_agreement = pd.DataFrame(agreement_results)

display(importance_exact_agreement)

In [ ]:
# I4 — Percentage within one point of the median
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/importance_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
df["rating_numeric"] = pd.to_numeric(
    df["rating_numeric"],
    errors="coerce"
)

df = df.dropna(
    subset=["respondent_id", "technology", "rating_numeric"]
)

within_one_results = []

for solution, solution_df in df.groupby("technology"):
    ratings = solution_df["rating_numeric"]
    solution_median = ratings.median()
    
    within_one = (ratings - solution_median).abs() <= 1
    
    within_one_results.append({
        "technology": solution,
        "n_experts": len(ratings),
        "median": solution_median,
        "n_within_one_point": int(within_one.sum()),
        "proportion_within_one_point": within_one.mean(),
        "percentage_within_one_point": 100 * within_one.mean()
    })

importance_within_one = pd.DataFrame(within_one_results)

display(importance_within_one)

In [ ]:
# I5 — Leave-one-expert-out stability of each median
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/importance_humanOnly_rq1.csv"

df = pd.read_csv(DATA)
df["rating_numeric"] = pd.to_numeric(
    df["rating_numeric"],
    errors="coerce"
)

df = df.dropna(
    subset=["respondent_id", "technology", "rating_numeric"]
)

loo_results = []

for solution, solution_df in df.groupby("technology"):
    solution_df = solution_df.reset_index(drop=True)
    full_median = solution_df["rating_numeric"].median()
    
    for removed_index, removed_row in solution_df.iterrows():
        remaining = solution_df.drop(index=removed_index)
        loo_median = remaining["rating_numeric"].median()
        
        loo_results.append({
            "technology": solution,
            "removed_expert": removed_row["respondent_id"],
            "removed_rating": removed_row["rating_numeric"],
            "full_n_experts": len(solution_df),
            "full_median": full_median,
            "loo_n_experts": len(remaining),
            "loo_median": loo_median,
            "median_changed": not np.isclose(
                loo_median,
                full_median
            )
        })

importance_loo_details = pd.DataFrame(loo_results)

importance_loo_summary = (
    importance_loo_details
    .groupby("technology")
    .agg(
        n_experts=("full_n_experts", "first"),
        full_median=("full_median", "first"),
        minimum_loo_median=("loo_median", "min"),
        maximum_loo_median=("loo_median", "max"),
        n_removals_changing_median=("median_changed", "sum"),
        total_leave_one_out_runs=("removed_expert", "count")
    )
    .reset_index()
)

importance_loo_summary["median_stable_under_all_removals"] = (
    importance_loo_summary["n_removals_changing_median"] == 0
)

display(importance_loo_summary)
display(importance_loo_details)

In [ ]:
# I6 — Ordinal Krippendorff’s alpha with bootstrap CI. This uses a stratified expert bootstrap. Experts are resampled within s1 and s2, preserving the incomplete survey design.
# the validation against the current data produced an observed alpha of approximately 0.084, indicating that this result may not support a claim of strong overall ordinal agreement. It must be interpreted alongside the strong ceiling effect and the solution-level consistency results.
from pathlib import Path
import numpy as np
import pandas as pd
import krippendorff

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/importance_humanOnly_rq1.csv"

N_BOOTSTRAP = 10_000
RANDOM_SEED = 20260728

df = pd.read_csv(DATA)
df["rating_numeric"] = pd.to_numeric(
    df["rating_numeric"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "respondent_id",
        "variant_id",
        "technology",
        "rating_numeric"
    ]
)

# Expert-by-solution matrix. Missing ratings remain NaN.
rating_matrix = df.pivot(
    index="respondent_id",
    columns="technology",
    values="rating_numeric"
)

observed_alpha = krippendorff.alpha(
    reliability_data=rating_matrix.to_numpy(dtype=float),
    level_of_measurement="ordinal"
)

# Identify the survey version completed by each expert.
rater_information = (
    df[["respondent_id", "variant_id"]]
    .drop_duplicates()
)

if rater_information["respondent_id"].duplicated().any():
    raise ValueError(
        "At least one expert is associated with multiple survey variants."
    )

variant_groups = {
    variant: group["respondent_id"].to_numpy()
    for variant, group in rater_information.groupby("variant_id")
}

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_alphas = []

for bootstrap_index in range(N_BOOTSTRAP):
    sampled_rows = []
    
    # Resample experts within each survey version.
    for variant, expert_ids in variant_groups.items():
        sampled_experts = rng.choice(
            expert_ids,
            size=len(expert_ids),
            replace=True
        )
        
        sampled_rows.extend(
            rating_matrix.loc[sampled_experts].to_numpy(dtype=float)
        )
    
    sampled_matrix = np.asarray(sampled_rows, dtype=float)
    
    try:
        bootstrap_alpha = krippendorff.alpha(
            reliability_data=sampled_matrix,
            level_of_measurement="ordinal"
        )
        
        if np.isfinite(bootstrap_alpha):
            bootstrap_alphas.append(bootstrap_alpha)
            
    except (ValueError, ZeroDivisionError, FloatingPointError):
        # Some bootstrap samples may not contain enough variation.
        continue

bootstrap_alphas = np.asarray(bootstrap_alphas)

if len(bootstrap_alphas) == 0:
    raise RuntimeError("No valid bootstrap alpha estimates were produced.")

ci_lower, ci_upper = np.quantile(
    bootstrap_alphas,
    [0.025, 0.975]
)

alpha_result = pd.DataFrame({
    "ordinal_krippendorff_alpha": [observed_alpha],
    "bootstrap_ci_lower_95": [ci_lower],
    "bootstrap_ci_upper_95": [ci_upper],
    "requested_bootstrap_samples": [N_BOOTSTRAP],
    "valid_bootstrap_samples": [len(bootstrap_alphas)]
})

display(alpha_result)

In [ ]:
# C. Barrier selection. B1 — Pairwise Jaccard coefficients. There is no unique ordinary Jaccard coefficient for 23 sets. This cell therefore reports the coefficient for every pair of experts, without calculating a mean or median.
from pathlib import Path
from itertools import combinations
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/barriers_human_final.csv"

df = pd.read_csv(DATA)
df["barrier_id"] = pd.to_numeric(
    df["barrier_id"],
    errors="coerce"
)

df = df.dropna(subset=["row_id", "barrier_id"])
df["barrier_id"] = df["barrier_id"].astype(int)

expert_sets = (
    df.groupby("row_id")["barrier_id"]
      .apply(set)
      .to_dict()
)

jaccard_results = []

for expert_1, expert_2 in combinations(sorted(expert_sets), 2):
    set_1 = expert_sets[expert_1]
    set_2 = expert_sets[expert_2]
    
    intersection_size = len(set_1.intersection(set_2))
    union_size = len(set_1.union(set_2))
    
    coefficient = (
        intersection_size / union_size
        if union_size > 0
        else float("nan")
    )
    
    jaccard_results.append({
        "expert_1": expert_1,
        "expert_2": expert_2,
        "intersection_size": intersection_size,
        "union_size": union_size,
        "jaccard_coefficient": coefficient
    })

jaccard_coefficients = pd.DataFrame(jaccard_results)

# Matrix presentation of the same pairwise coefficients.
expert_ids = sorted(expert_sets)

jaccard_matrix = pd.DataFrame(
    1.0,
    index=expert_ids,
    columns=expert_ids
)

for row in jaccard_coefficients.itertuples(index=False):
    jaccard_matrix.loc[
        row.expert_1,
        row.expert_2
    ] = row.jaccard_coefficient
    
    jaccard_matrix.loc[
        row.expert_2,
        row.expert_1
    ] = row.jaccard_coefficient

display(jaccard_coefficients)
display(jaccard_matrix.round(3))

In [ ]:
# B2 — Selection rate for each barrier using all experts. The denominator is the number of experts, not the total number of selections.

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/barriers_human_final.csv"

df = pd.read_csv(DATA)
df["barrier_id"] = pd.to_numeric(
    df["barrier_id"],
    errors="coerce"
)

df = df.dropna(subset=["row_id", "barrier_id"])
df["barrier_id"] = df["barrier_id"].astype(int)

# Prevent duplicate selection of the same barrier by one expert.
df = df.drop_duplicates(subset=["row_id", "barrier_id"])

expert_ids = sorted(df["row_id"].unique())
n_experts = len(expert_ids)

all_expert_rates = []

for barrier_id in range(1, 12):
    n_selecting = df.loc[
        df["barrier_id"] == barrier_id,
        "row_id"
    ].nunique()
    
    all_expert_rates.append({
        "barrier_id": barrier_id,
        "n_experts": n_experts,
        "n_selecting": n_selecting,
        "selection_rate": n_selecting / n_experts,
        "selection_percentage": 100 * n_selecting / n_experts
    })

barrier_selection_rates = pd.DataFrame(all_expert_rates)

display(barrier_selection_rates)

In [ ]:
# B3 — Leave-one-expert-out selection rates. This reports the recalculated rate for every omitted expert and every barrier.

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/HP/Documents/second-publication/current-analysis/re-run/rq1-rerun"
)
DATA = ROOT / "raw-responses/barriers_human_final.csv"

df = pd.read_csv(DATA)
df["barrier_id"] = pd.to_numeric(
    df["barrier_id"],
    errors="coerce"
)

df = df.dropna(subset=["row_id", "barrier_id"])
df["barrier_id"] = df["barrier_id"].astype(int)

df = df.drop_duplicates(subset=["row_id", "barrier_id"])

expert_sets = (
    df.groupby("row_id")["barrier_id"]
      .apply(set)
      .to_dict()
)

expert_ids = sorted(expert_sets)
n_all_experts = len(expert_ids)

# Full-sample rate for comparison with each LOO rate.
full_rates = {
    barrier_id: (
        sum(
            barrier_id in expert_sets[expert_id]
            for expert_id in expert_ids
        ) / n_all_experts
    )
    for barrier_id in range(1, 12)
}

loo_results = []

for omitted_expert in expert_ids:
    retained_experts = [
        expert_id
        for expert_id in expert_ids
        if expert_id != omitted_expert
    ]
    
    n_retained = len(retained_experts)
    
    for barrier_id in range(1, 12):
        n_selecting = sum(
            barrier_id in expert_sets[expert_id]
            for expert_id in retained_experts
        )
        
        loo_rate = n_selecting / n_retained
        
        loo_results.append({
            "omitted_expert": omitted_expert,
            "barrier_id": barrier_id,
            "full_sample_rate": full_rates[barrier_id],
            "loo_n_experts": n_retained,
            "loo_n_selecting": n_selecting,
            "loo_selection_rate": loo_rate,
            "change_from_full_rate": (
                loo_rate - full_rates[barrier_id]
            )
        })

barrier_loo_rates = pd.DataFrame(loo_results)

# Matrix: rows are omitted experts and columns are barriers.
barrier_loo_matrix = barrier_loo_rates.pivot(
    index="omitted_expert",
    columns="barrier_id",
    values="loo_selection_rate"
)

barrier_loo_matrix.columns = [
    f"B{barrier_id}"
    for barrier_id in barrier_loo_matrix.columns
]

display(barrier_loo_rates)
display(barrier_loo_matrix.round(3))